In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target_classification, get_features_and_target
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import train_test_split

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data_classifier.csv")
dev_df = pd.read_csv("data/development_data_classifier.csv")

sc = StandardScaler()

target_column = ("Category")  

x_train, y_train = get_features_and_target_classification(train_df, target_column)
x_dev, y_dev = get_features_and_target_classification(dev_df, target_column)


In [4]:
x_train

,Sample ID,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,1,35,200,0,0.00,1315.41,0.922,0.920
1,1,35,200,0,3.41,1337.45,0.922,0.920
2,1,35,200,0,6.82,1081.47,0.922,0.920
3,2,35,1500,0,0.00,1819.13,0.920,0.925
4,2,35,1500,0,3.41,2016.44,0.920,0.925
...,...,...,...,...,...,...,...,...
2448,494,60,1200,0,96.62,3400.24,0.620,0.624
2449,494,60,1200,0,96.55,3855.83,0.620,0.624
2450,494,60,1200,0,96.48,4173.30,0.620,0.624
2451,494,60,1200,0,96.41,3485.17,0.620,0.624


# Add Physical Columns Interfacial_Failure and Pullout_Failure

In [5]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     x = df[['Thickness A (mm)', 'Thickness B (mm)']].sum(axis=1)
     # x can be approximated to metal sheet thickness. Change 2*t either to t to use the thinner 
     # metal sheet or 2*x to test if the sum of both metal sheets give beter results
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

In [6]:
x_dev

,Sample ID,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,6,95,1500,0,0.00,1213.38,0.918,0.925
1,6,95,1500,0,8.29,1230.27,0.918,0.925
2,6,95,1500,0,16.57,1217.63,0.918,0.925
3,6,95,1500,0,24.85,1111.28,0.918,0.925
4,6,95,1500,0,33.15,1126.65,0.918,0.925
...,...,...,...,...,...,...,...,...
835,489,60,1200,0,98.26,3576.75,0.625,0.624
836,489,60,1200,0,98.22,3024.94,0.625,0.624
837,489,60,1200,0,98.18,2984.87,0.625,0.624
838,489,60,1200,0,98.14,2950.68,0.625,0.624


# Fit Model

In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score

# columns that vary within a sample
time_series_cols = ["Force (N)", "Current (A)"]

# static columns
static_cols = ["Pressure (PSI)", "Welding Time (ms)", "Angle (Deg)", "Thickness A (mm)", "Thickness B (mm)"]

agg_df_train = train_df.groupby("Sample ID")[time_series_cols].agg(
    ['mean', 'std', 'min', 'max']
)
agg_df_dev = dev_df.groupby("Sample ID")[time_series_cols].agg(
    ['mean', 'std', 'min', 'max']
)

# flatten multi-index columns
agg_df_train.columns = ['_'.join(col) for col in agg_df_train.columns]
agg_df_dev.columns = ['_'.join(col) for col in agg_df_dev.columns]

# --- Add static columns (take first value per sample) --- 
static_df_train = train_df.groupby("Sample ID")[static_cols].first() 
static_df_dev = dev_df.groupby("Sample ID")[static_cols].first() 

 # Total Welding Time, adds the Welding Time Cycles in a single sample
weld_time_sum_train = train_df.groupby("Sample ID")["Welding Time (ms)"].sum() 
weld_time_sum_train = weld_time_sum_train.rename("Welding_Time_Total") 
weld_time_sum_dev = dev_df.groupby("Sample ID")["Welding Time (ms)"].sum() 
weld_time_sum_dev = weld_time_sum_dev.rename("Welding_Time_Total") 

# Combination of datasets
join_static_df_train = agg_df_train.join(static_df_train) 
x_train = join_static_df_train.join(weld_time_sum_train) 
join_static_df_dev = agg_df_dev.join(static_df_dev) 
x_dev = join_static_df_dev.join(weld_time_sum_dev) 

# Target value
y_train = train_df.groupby("Sample ID")['Category'].first()

try:
    x_train_scaled = sc.fit_transform(X=x_train)
    x_dev_scaled = sc.transform(x_dev)

    classifier = TabPFNClassifier() 

    classifier.fit(x_train_scaled,y_train)

    predictions_class_train = classifier.predict(x_train_scaled)
    predictions_class_dev = classifier.predict(x_dev_scaled)

except Exception as e:
    print(f"No Scaling this time")

    classifier = TabPFNClassifier() 

    classifier.fit(x_train,y_train)

    predictions_class_train = classifier.predict(x_train)
    predictions_class_dev = classifier.predict(x_dev)



In [31]:
df_mean = pd.concat([x_train, x_dev], axis=0)


In [33]:
df_mean.to_csv("data/df_mean.csv", index=False)

# Add Pullforce as Target

In [9]:
def compute_pullforces(x, y, predictions_class):
    pullforces = []

    for sample_id, pred in zip(y.index, predictions_class):
        row_x_df = x.loc[[sample_id]]  # 1-row DataFrame

        if pred == "Bad":
            value = compute_interfacial_failure(row_x_df).iloc[0]
        else:
            value = compute_pullout_failure(row_x_df).iloc[0]

        pullforces.append(value)

    return np.array(pullforces)


In [10]:
y_train = train_df.groupby("Sample ID")['Category'].first()
predictions_class_train = classifier.predict(x_train)

pullforces_train = compute_pullforces(x_train, y_train, predictions_class_train)

y_dev = dev_df.groupby("Sample ID")['Category'].first()
predictions_class_dev = classifier.predict(x_dev)

pullforces_dev = compute_pullforces(x_dev, y_dev, predictions_class_dev)


/home/HTW-AALEN/76834/.conda/envs/phy_based_mlearning/lib/python3.10/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNClassifier was fitted without feature names
  warnings.warn(
/home/HTW-AALEN/76834/.conda/envs/phy_based_mlearning/lib/python3.10/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNClassifier was fitted without feature names
  warnings.warn(


In [11]:
print(predictions_class_dev)

['Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good']


In [12]:
print(pullforces_dev)

[5967.  6010.2 6042.8 6064.5 6097.2 6064.5 6097.2 6206.6 3423.9 3170.5
 3245.7 3237.3 3254.1 3304.7 3195.5 3245.7 3153.9 3203.8 3087.8 3330.1
 3245.7 3270.9 3162.2 3187.1 3220.5 3220.5 3145.6 3187.1 3129.  3162.2
 3096.  3112.5 3220.5 3228.9 3096.  3254.1 3270.9 3137.3 3170.5 3212.2
 3137.3 3254.1 3228.9 3145.6 3212.2 3220.5 3254.1 3262.5 3145.6 3071.3
 3055.  3038.6 3178.8 3129.  3162.2 3145.6 3170.5 3120.8 3162.2 3178.8
 3153.9 3104.3 3137.3 3153.9 3129.  3137.3 3096.  3055.  3079.6 3137.3
 3104.3 3220.5 3162.2 3178.8 3112.5 3104.3 3137.3 3104.3 3129.  3187.1
 3212.2 3145.6 3195.5 3170.5 3170.5 3112.5 3120.8 3162.2 3153.9 3170.5
 3038.6 3112.5 3153.9 3071.3 3153.9 3145.6 3170.5 3112.5 3153.9]


In [61]:
y_dev_Regressor = dev_df.groupby("Sample ID")['PullTest (N)'].first()
y_dev_delta = pullforces_dev - y_dev_Regressor

y_train_Regressor = train_df.groupby("Sample ID")['PullTest (N)'].first()
y_train_delta =  pullforces_train - y_train_Regressor

In [44]:
y_train_Regressor.head(10)

Sample ID
1     2127.7
2     5346.4
4     2350.4
5     2174.8
7     3897.5
9     1932.5
11    3816.0
12    1832.5
15    5634.8
17    2685.9
Name: PullTest (N), dtype: float64

In [15]:
pullforces_train

array([5988.6, 5988.6, 5902.3, 6195.6, 6097.2, 6075.4, 6064.5, 5934.6,
       6108.1, 6162.7, 6042.8, 6042.8, 6075.4, 6162.7, 6283.7, 6305.8,
       6031.9, 3195.5, 3270.9, 3279.3, 3178.8, 3321.6, 3237.3, 3153.9,
       3120.8, 3203.8, 3178.8, 3195.5, 3287.8, 3228.9, 3270.9, 3245.7,
       3212.2, 3212.2, 3212.2, 3212.2, 3120.8, 3279.3, 3096. , 3137.3,
       3087.8, 3153.9, 3220.5, 3287.8, 3178.8, 3203.8, 3262.5, 3162.2,
       3187.1, 3145.6, 3220.5, 3145.6, 3237.3, 3162.2, 3220.5, 3245.7,
       3170.5, 3203.8, 3262.5, 3270.9, 3254.1, 3220.5, 3237.3, 3237.3,
       3237.3, 3220.5, 3338.6, 3228.9, 3220.5, 3245.7, 3178.8, 3145.6,
       3145.6, 3262.5, 3162.2, 3162.2, 3228.9, 3195.5, 3137.3, 3237.3,
       3254.1, 3245.7, 3120.8, 3087.8, 3112.5, 3104.3, 3071.3, 3104.3,
       3120.8, 3087.8, 3055. , 3087.8, 3079.6, 3245.7, 3195.5, 3162.2,
       3304.7, 3145.6, 3079.6, 3104.3, 3212.2, 3245.7, 3262.5, 3262.5,
       3212.2, 3237.3, 3195.5, 3237.3, 3220.5, 3212.2, 3212.2, 3162.2,
      

In [51]:
y_train_delta.head(10)

Sample ID
1    -3860.9
2     -642.2
4    -3551.9
5    -4020.8
7    -2199.7
9    -4142.9
11   -2248.5
12   -4102.1
15    -473.3
17   -3476.8
Name: PullTest (N), dtype: float64

In [17]:
x_train

,Force (N)_mean,Force (N)_std,Force (N)_min,Force (N)_max,Current (A)_mean,Current (A)_std,Current (A)_min,Current (A)_max,Pressure (PSI),Welding Time (ms),Angle (Deg),Thickness A (mm),Thickness B (mm),Welding_Time_Total
Sample ID,,,,,,,,,,,,,,
1,3.410000,3.410000,0.00,6.82,1244.776667,141.856410,1081.47,1337.45,35,200,0,0.922,0.920,600
2,25.967500,16.599982,0.00,52.25,1840.571250,98.110791,1709.96,2016.44,35,1500,0,0.920,0.925,24000
4,8.286667,8.285001,0.00,16.57,1416.673333,90.008858,1321.93,1501.05,95,200,0,0.912,0.924,600
5,33.136667,8.285001,24.85,41.42,1474.350000,182.155941,1268.82,1615.83,95,200,0,0.948,0.939,600
7,97.644667,21.377203,63.82,127.46,1035.198000,92.864686,871.91,1150.82,35,1500,0,0.930,0.937,22500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,98.536154,0.083520,98.34,98.63,3916.430000,529.220757,2730.90,4554.81,60,1200,0,0.625,0.622,15600
490,97.889231,0.283092,97.32,98.15,2803.468462,276.735971,2542.37,3369.76,60,1200,0,0.622,0.632,15600
492,98.490833,0.277700,98.07,98.87,3625.364167,488.348727,2495.94,4337.00,60,1200,0,0.666,0.633,14400


# Fit 2nd Model

In [62]:
# Initialize the regressor
regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
try:
    x_train_scaled = sc.fit_transform(X=x_train)
    x_dev_scaled = sc.transform(x_dev)

    regressor.fit(x_train_scaled, y_train_delta)

    # Predict on the test set
    predictions_class_dev = regressor.predict(x_dev_scaled)
    
except Exception as e:
    print(f"No Scaling: {e}")

    regressor.fit(x_train, y_train_delta)

    # Predict on the test set
    predictions_class_dev = regressor.predict(x_dev)
    

final_pred = pullforces_dev + predictions_class_dev

pred_series = pd.Series(predictions_class_dev, index=x_dev.index)
pull_series = pd.Series(pullforces_dev, index=y_dev.index)

final_pred = pull_series.loc[pred_series.index] - pred_series


In [19]:
y_dev.index

Index([  6,   8,  14,  16,  22,  25,  28,  32,  34,  40,  41,  56,  62,  65,
        74,  77,  79,  83,  88,  91,  92,  93, 101, 102, 105, 123, 133, 136,
       140, 143, 145, 150, 179, 182, 188, 190, 193, 198, 212, 220, 221, 225,
       226, 236, 249, 262, 264, 265, 276, 277, 282, 283, 285, 294, 295, 296,
       298, 301, 302, 306, 314, 319, 330, 336, 338, 349, 351, 354, 356, 359,
       360, 365, 366, 372, 376, 377, 383, 393, 405, 414, 415, 419, 421, 422,
       423, 432, 433, 434, 442, 443, 449, 452, 457, 459, 468, 471, 474, 475,
       489],
      dtype='int64', name='Sample ID')

In [20]:
x_dev.index

Index([  6,   8,  14,  16,  22,  25,  28,  32,  34,  40,  41,  56,  62,  65,
        74,  77,  79,  83,  88,  91,  92,  93, 101, 102, 105, 123, 133, 136,
       140, 143, 145, 150, 179, 182, 188, 190, 193, 198, 212, 220, 221, 225,
       226, 236, 249, 262, 264, 265, 276, 277, 282, 283, 285, 294, 295, 296,
       298, 301, 302, 306, 314, 319, 330, 336, 338, 349, 351, 354, 356, 359,
       360, 365, 366, 372, 376, 377, 383, 393, 405, 414, 415, 419, 421, 422,
       423, 432, 433, 434, 442, 443, 449, 452, 457, 459, 468, 471, 474, 475,
       489],
      dtype='int64', name='Sample ID')

In [56]:
pred_series

Sample ID
6     -1558.495483
8     -3944.052734
14    -1582.276123
16    -3941.896484
22     -933.993286
          ...     
468    -201.094238
471    -206.451080
474    -202.137634
475    -198.543518
489    -594.045593
Length: 99, dtype: float32

In [22]:
pull_series

Sample ID
6      5967.0
8      6010.2
14     6042.8
16     6064.5
22     6097.2
        ...  
468    3153.9
471    3145.6
474    3170.5
475    3112.5
489    3153.9
Length: 99, dtype: float64

In [23]:
pullforces_dev

array([5967. , 6010.2, 6042.8, 6064.5, 6097.2, 6064.5, 6097.2, 6206.6,
       3423.9, 3170.5, 3245.7, 3237.3, 3254.1, 3304.7, 3195.5, 3245.7,
       3153.9, 3203.8, 3087.8, 3330.1, 3245.7, 3270.9, 3162.2, 3187.1,
       3220.5, 3220.5, 3145.6, 3187.1, 3129. , 3162.2, 3096. , 3112.5,
       3220.5, 3228.9, 3096. , 3254.1, 3270.9, 3137.3, 3170.5, 3212.2,
       3137.3, 3254.1, 3228.9, 3145.6, 3212.2, 3220.5, 3254.1, 3262.5,
       3145.6, 3071.3, 3055. , 3038.6, 3178.8, 3129. , 3162.2, 3145.6,
       3170.5, 3120.8, 3162.2, 3178.8, 3153.9, 3104.3, 3137.3, 3153.9,
       3129. , 3137.3, 3096. , 3055. , 3079.6, 3137.3, 3104.3, 3220.5,
       3162.2, 3178.8, 3112.5, 3104.3, 3137.3, 3104.3, 3129. , 3187.1,
       3212.2, 3145.6, 3195.5, 3170.5, 3170.5, 3112.5, 3120.8, 3162.2,
       3153.9, 3170.5, 3038.6, 3112.5, 3153.9, 3071.3, 3153.9, 3145.6,
       3170.5, 3112.5, 3153.9])

In [55]:
final_pred

Sample ID
6       7525.495483
8       9954.252734
14      7625.076123
16     10006.396484
22      7031.193286
           ...     
468     3354.994238
471     3352.051080
474     3372.637634
475     3311.043518
489     3747.945593
Length: 99, dtype: float64

In [25]:
y_dev

Sample ID
6         Good
8          Bad
14        Good
16         Bad
22     Explode
        ...   
468       Good
471       Good
474       Good
475       Good
489    Explode
Name: Category, Length: 99, dtype: object

# Check Validation Data

In [63]:
y_dev = dev_df.groupby("Sample ID")['PullTest (N)'].first()
y_dev

# Convert to numpy arrays
true_vals = np.array(y_dev).ravel()
pred_vals = np.array(final_pred).ravel()

# Sample index
sample_idx = np.arange(len(true_vals))

# Extract category per sample (aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

# Masks
mask_good    = categories == "Good"
mask_bad     = categories == "Bad"
mask_explode = categories == "Explode"

fig = go.Figure()

# --- GOOD (circles) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=true_vals[mask_good],
    mode="markers",
    name="Good (True)",
    marker=dict(symbol="circle", color="red", size=7)
))
fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=pred_vals[mask_good],
    mode="markers",
    name="Good (Pred)",
    marker=dict(symbol="circle", color="blue", size=7)
))

# --- BAD (X) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=true_vals[mask_bad],
    mode="markers",
    name="Bad (True)",
    marker=dict(symbol="x", color="red", size=9)
))
fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=pred_vals[mask_bad],
    mode="markers",
    name="Bad (Pred)",
    marker=dict(symbol="x", color="blue", size=9)
))

# --- EXPLODE (triangle-up) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=true_vals[mask_explode],
    mode="markers",
    name="Explode (True)",
    marker=dict(symbol="triangle-up", color="red", size=9)
))
fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=pred_vals[mask_explode],
    mode="markers",
    name="Explode (Pred)",
    marker=dict(symbol="triangle-up", color="blue", size=9)
))

# Connecting lines
for i in range(len(sample_idx)):
    fig.add_trace(go.Scatter(
        x=[sample_idx[i], sample_idx[i]],
        y=[true_vals[i], pred_vals[i]],
        mode="lines",
        line=dict(color="gray", width=1),
        showlegend=False
    ))

fig.update_layout(
    title="Validation Samples: True vs Prediction (TabPFN) by Category",
    xaxis_title="Sample Index",
    yaxis_title="Pull Force",
    template="seaborn"
)

fig.write_html("Graphs/TabPFN_Evaltrue.html")
fig.show()



In [27]:
y_dev.tail(94)

Sample ID
25     2867.4
28     3855.5
32     2764.4
34     3281.1
40     3256.1
        ...  
468    2865.5
471    2860.7
474    2902.6
475    3000.7
489    2054.5
Name: PullTest (N), Length: 94, dtype: float64

In [28]:
final_pred

Sample ID
6      4408.504517
8      2066.147266
14     4460.523877
16     2122.603516
22     5163.206714
          ...     
468    2952.805762
471    2939.148920
474    2968.362366
475    2913.956482
489    2559.854407
Length: 99, dtype: float64

# Check Validation Loss and R2

In [64]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, final_pred)
rmse = np.sqrt(mean_squared_error(y_dev, final_pred))
R2   = r2_score(y_dev, final_pred)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")

MAE:  133.13
RMSE: 217.83
R2: 0.63

MAE:  133.63
RMSE: 215.43
R2: 0.63
